In [5]:

!git clone https://github.com/MohamedElsayed75/FRW1NB1.git
%cd FRW1NB1/work/notebooks

Cloning into 'FRW1NB1'...
remote: Enumerating objects: 114, done.
remote: Counting objects: 100% (114/114), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 114 (delta 31), reused 79 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (114/114), 1.85 MiB | 4.66 MiB/s, done.
Resolving deltas: 100% (31/31), done.
/content/FRW1NB1/work/notebooks


In [2]:
import duckdb

con = duckdb.connect()

con.execute(f"""
INSTALL httpfs;
LOAD httpfs;

CREATE OR REPLACE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

print("DuckDB/Hugging Face connection ready.")

DuckDB/Hugging Face connection ready.


In [7]:
TABLE = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

print(TABLE)

hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet


In [8]:
df_columns = con.execute(f"""
    DESCRIBE SELECT *
    FROM read_parquet('{TABLE}')
    LIMIT 1
""").fetchdf()

df_columns

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Contract

**1. What one row means:**  
One row represents one content item for one client on one reporting date.

**2. Tables I use:**  
I use `fact_content_daily_performance`, which contains daily Search Console and Analytics performance for content items.

**3. Time window:**  
I use March 2026 as the development month. I use a mid-panel month rather than the June 2026 `_sample` month so that June can remain a sealed outcome/test month.

**4. What I predict/rank:**  
My lane is Refresh / Content Opportunity Scoring. I rank content items by whether their performance is declining. For this exercise, I use the month-over-month change in organic/search performance to construct a decline label.

**5. What I deliberately exclude:**  
I deliberately exclude future/outcome-derived fields from the feature set. In particular, I do not use the outcome used to construct the decline label as a model feature.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field buckets

**Candidate features:**
- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `ga4_pageviews`
- `ga4_sessions`

**Label / outcome:**
- A derived decline indicator based on the content item's later performance.

**Context:**
- `client_hash_id`
- `content_hash_id`
- `report_date`
- `month`

**Availability fields:**
- `gsc_data_available`
- `ga4_data_available`

**Excluded:**
- The derived decline label itself.
- `client_hash_id` and `content_hash_id` as predictive features because they are identifiers.
- Any future-period performance used to construct the outcome.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1 — Grain

I test whether `(client_hash_id, content_hash_id, report_date)` occurs more than once in March 2026. If the query returns zero rows, this supports the stated grain of one content item × one client × one reporting date.

In [9]:
grain_check = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS rows_at_grain
FROM read_parquet('{TABLE}')
WHERE month = '2026-03'
GROUP BY client_hash_id, content_hash_id, report_date
HAVING COUNT(*) > 1
LIMIT 5
""").fetchdf()

print("Duplicate grain combinations found:", len(grain_check))
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain combinations found: 0


,client_hash_id,content_hash_id,report_date,rows_at_grain


### Query 2 — March row count and date span

I measure the number of rows in the March 2026 slice and its observed date range.

In [10]:
march_summary = con.execute(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM read_parquet('{TABLE}')
WHERE month = '2026-03'
""").fetchdf()

march_summary

,row_count,min_report_date,max_report_date
0,9841378,2026-03-01,2026-03-31


### Query 3 — Data availability

I check how many March rows have Search Console and Analytics data explicitly marked as available. I use `IS TRUE` rather than assuming that zero-valued metrics mean that data is available.

In [11]:
availability_check = con.execute(f"""
SELECT
    COUNT(*) AS march_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS rows_with_gsc_available,
    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS rows_with_ga4_available
FROM read_parquet('{TABLE}')
WHERE month = '2026-03'
""").fetchdf()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_rows,rows_with_gsc_available,rows_with_ga4_available
0,9841378,3611061,413966


## Five-feature frame

I use five performance features from the March 2026 development slice. These are measurements available at the observation/decision moment and are not derived from the decline outcome.

In [12]:
feature_frame = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,

    AVG(gsc_impressions) AS gsc_impressions,
    AVG(gsc_clicks) AS gsc_clicks,
    AVG(gsc_avg_position) AS gsc_avg_position,
    AVG(ga4_pageviews) AS ga4_pageviews,
    AVG(ga4_sessions) AS ga4_sessions

FROM read_parquet('{TABLE}')
WHERE month = '2026-03'

GROUP BY
    client_hash_id,
    content_hash_id
""").fetchdf()

print("Feature frame shape:", feature_frame.shape)

feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (331437, 7)


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,210.419355,0.225806,7.209549,0.090909,0.090909
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,14.612903,0.000000,2.987198,0.000000,0.000000
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,181.612903,0.193548,6.724039,0.545455,0.272727
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,159.483871,0.419355,7.244844,0.181818,0.181818
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,1.354839,0.000000,14.432540,0.727273,0.636364


### Available when?

| Feature | Available when? |
|---|---|
| `gsc_impressions` | Available from Search Console observations before the decision/outcome moment. |
| `gsc_clicks` | Available from Search Console observations before the decision/outcome moment. |
| `gsc_avg_position` | Available from Search Console observations before the decision/outcome moment; zero/no-data cases are not interpreted as a valid ranking position. |
| `ga4_pageviews` | Available before the decision/outcome moment when `ga4_data_available IS TRUE`. |
| `ga4_sessions` | Available before the decision/outcome moment when `ga4_data_available IS TRUE`. |

## Deliberate leakage experiment

For this exercise, I define a decline label using month-over-month organic search clicks. A content item is labelled as declining when its March average GSC clicks are lower than its February average.

I then deliberately create a feature directly from this label to demonstrate target leakage.

In [13]:
label_df = con.execute(f"""
WITH monthly AS (
    SELECT
        client_hash_id,
        content_hash_id,
        month,
        AVG(gsc_clicks) AS avg_gsc_clicks
    FROM read_parquet('{TABLE}')
    WHERE month IN ('2026-02', '2026-03')
    GROUP BY
        client_hash_id,
        content_hash_id,
        month
),

pivoted AS (
    SELECT
        client_hash_id,
        content_hash_id,
        MAX(CASE WHEN month = '2026-02'
            THEN avg_gsc_clicks END) AS feb_clicks,
        MAX(CASE WHEN month = '2026-03'
            THEN avg_gsc_clicks END) AS mar_clicks
    FROM monthly
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    client_hash_id,
    content_hash_id,
    feb_clicks,
    mar_clicks,
    CASE
        WHEN feb_clicks IS NOT NULL
         AND mar_clicks IS NOT NULL
         AND mar_clicks < feb_clicks
        THEN 1
        ELSE 0
    END AS is_declining_label
FROM pivoted
WHERE feb_clicks IS NOT NULL
  AND mar_clicks IS NOT NULL
""").fetchdf()

label_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,feb_clicks,mar_clicks,is_declining_label
0,client_3ffa76342f366962,content_feccf822ac21326e,0.0,0.0,0
1,client_3ffa76342f366962,content_17cf93c10413ebe9,0.0,0.0,0
2,client_3ffa76342f366962,content_14aa55a733a83e24,0.0,0.0,0
3,client_3ffa76342f366962,content_1ffdc7d0f4c3c5b8,0.0,0.0,0
4,client_3ffa76342f366962,content_a0d0fcdb5075e697,0.0,0.0,0


In [14]:
model_df = feature_frame.merge(
    label_df[
        [
            "client_hash_id",
            "content_hash_id",
            "is_declining_label"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Model frame shape:", model_df.shape)
model_df.head()

Model frame shape: (303572, 8)


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,is_declining_label
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,210.419355,0.225806,7.209549,0.090909,0.090909,1
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,14.612903,0.000000,2.987198,0.000000,0.000000,1
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,181.612903,0.193548,6.724039,0.545455,0.272727,0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,159.483871,0.419355,7.244844,0.181818,0.181818,1
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,1.354839,0.000000,14.432540,0.727273,0.636364,0


In [15]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

leak_df = model_df.copy()

# DELIBERATE TARGET LEAK
leak_df["leaked_label"] = leak_df["is_declining_label"]

leaky_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "leaked_label"
]

X = leak_df[leaky_features]
y = leak_df["is_declining_label"]

imputer = SimpleImputer(strategy="median")
X = imputer.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

leaky_score = roc_auc_score(
    y_test,
    model.predict_proba(X_test)[:, 1]
)

print(f"Leaky ROC-AUC: {leaky_score:.4f}")

Leaky ROC-AUC: 1.0000


### Leakage result

The leaked model achieves an unrealistically high ROC-AUC because `leaked_label` is directly copied from the target. The model has effectively been given the answer.

This high score is therefore evidence of target leakage, not evidence of a useful predictive model.

The leaked feature must be removed because the label is not available at the decision moment.

In [16]:
SAFE_FEATURES = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions"
]

honest_X = model_df[SAFE_FEATURES]
honest_y = model_df["is_declining_label"]

imputer = SimpleImputer(strategy="median")
honest_X = imputer.fit_transform(honest_X)

X_train, X_test, y_train, y_test = train_test_split(
    honest_X,
    honest_y,
    test_size=0.30,
    random_state=42,
    stratify=honest_y
)

honest_model = LogisticRegression(max_iter=1000)
honest_model.fit(X_train, y_train)

honest_score = roc_auc_score(
    y_test,
    honest_model.predict_proba(X_test)[:, 1]
)

print(f"Honest ROC-AUC: {honest_score:.4f}")

Honest ROC-AUC: 0.8061


In [17]:
final_feature_frame = model_df[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_pageviews",
        "ga4_sessions",
        "is_declining_label"
    ]
].copy()

print("Final feature columns:")
print(SAFE_FEATURES)

print("\nIs leaked_label in final feature frame?")
print("leaked_label" in final_feature_frame.columns)

final_feature_frame.head()

Final feature columns:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions']

Is leaked_label in final feature frame?
False


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,is_declining_label
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,210.419355,0.225806,7.209549,0.090909,0.090909,1
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,14.612903,0.000000,2.987198,0.000000,0.000000,1
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,181.612903,0.193548,6.724039,0.545455,0.272727,0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,159.483871,0.419355,7.244844,0.181818,0.181818,1
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,1.354839,0.000000,14.432540,0.727273,0.636364,0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Limitation

A limitation of this slice is that the decline label depends on having both February and March observations for the same content item. Content with incomplete history cannot be labelled reliably using this month-over-month definition and is therefore excluded from the labelled modelling frame.

GA4 availability also varies between clients, so Analytics-based features are not equally informative for every content item.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [18]:
print("SELF-CHECK")
print("===========")

checks = {
    "Five contract answers completed": True,
    "Exactly three verification queries": True,
    "Grain checked": len(grain_check) == 0,
    "March count and date span checked": not march_summary.empty,
    "Availability checked with IS TRUE": not availability_check.empty,
    "Exactly five features": len(SAFE_FEATURES) == 5,
    "Leakage experiment performed": "leaked_label" in leak_df.columns,
    "Leaked feature removed": "leaked_label" not in final_feature_frame.columns,
    "Limitation documented": True,
}

for name, passed in checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")

SELF-CHECK
[PASS] Five contract answers completed
[PASS] Exactly three verification queries
[PASS] Grain checked
[PASS] March count and date span checked
[PASS] Availability checked with IS TRUE
[PASS] Exactly five features
[PASS] Leakage experiment performed
[PASS] Leaked feature removed
[PASS] Limitation documented
